In [ ]:
import os
import numpy as np
import rawpy as rp
import imageio
import torch
from tqdm import tqdm


def load_npy(filepath):
    """
    Load .npy image and normalize to [0, 1].
    """
    img = np.load(filepath)
    img = img / 1023.0  
    return img


def save_4ch_npy_png(tensor_to_save, res_dir, fname, save_type, background_dng):
    """
    Insert 4-channel npy tensor into Bayer pattern of background DNG
    and save as processed PNG.
    """
    file_basename = os.path.basename(fname).replace(".npy", "")
    postfix = f"_{save_type}.png"
    output_path = os.path.join(res_dir, save_type, file_basename + postfix)

    # Skip if file already exists
    if os.path.exists(output_path):
        print(f"Skipping {output_path} (already exists)")
        return

    # Create output directory
    os.makedirs(os.path.join(res_dir, save_type), exist_ok=True)

    # Convert tensor to numpy and rescale
    npy = tensor_to_save.to("cpu").detach().numpy()
    npy = npy.squeeze() * 1023.0  # back to 10-bit scale

    if npy.ndim != 3 or npy.shape[0] != 4:
        raise ValueError(f"Expected a (4, H, W) array, got shape {npy.shape}")

    # Load DNG background
    data = rp.imread(background_dng)

    # Extract Bayer channels
    GR = data.raw_image[0::2, 0::2]
    R = data.raw_image[0::2, 1::2]
    B = data.raw_image[1::2, 0::2]
    GB = data.raw_image[1::2, 1::2]

    # Zero them out first
    GR[:, :] = 0
    R[:, :] = 0
    B[:, :] = 0
    GB[:, :] = 0

    # Get dimensions
    w, h = npy.shape[1:]
    H, W = GR.shape

    # --- FIX: Crop or truncate to fit DNG dimensions ---
    min_h = min(H, w)
    min_w = min(W, h)

    GR[:min_h, :min_w] = npy[0][:min_h, :min_w]
    R[:min_h, :min_w] = npy[1][:min_h, :min_w]
    B[:min_h, :min_w] = npy[2][:min_h, :min_w]
    GB[:min_h, :min_w] = npy[3][:min_h, :min_w]

    # Postprocess raw to RGB
    newData = data.postprocess()

    # Crop region of interest
    start = (0, 464)
    end = (3584, 3024)
    newData = newData[start[0]:end[0], start[1]:end[1]]

    # Save PNG
    imageio.imwrite(output_path, newData)
    print(f"Saved: {output_path}")


if __name__ == "__main__":
    # Input file paths
    gt_path = "./GT.npy"
    input_path = "./input.npy"
    background_dng = "./background.dng"

    # Output directory
    res_dir = "./results_png"

    # Process both files
    for fname, tag in zip([gt_path, input_path], ["GT", "input"]):
        print(f"Processing {tag} image...")
        npy_data = np.float32(load_npy(fname))

        # Convert to tensor (H, W, C) → (C, H, W)
        if npy_data.ndim == 3 and npy_data.shape[-1] == 4:
            tensor = torch.from_numpy(npy_data).permute(2, 0, 1)
        elif npy_data.ndim == 3 and npy_data.shape[0] == 4:
            tensor = torch.from_numpy(npy_data)
        else:
            raise ValueError(f"Unexpected .npy shape: {npy_data.shape}")

        tensor = torch.clamp(tensor, 0, 1)

        # Save as PNG using DNG background
        save_4ch_npy_png(tensor, res_dir, fname, tag, background_dng)


Processing GT image...
Saved: ./results_png/GT/GT_GT.png
Processing input image...
Saved: ./results_png/input/input_input.png
